# Spark en MacBook Air M4 · 24 GB
Dos workers, dos núcleos y 2 GiB de memoria Spark por worker. Ejecuta una sola aplicación a la vez.

# Tu primer clúster de Spark
SBD · IABD · IES Ataúlfo Argenta

Ejecuta las celdas en orden. Mantén abierto el master en http://localhost:8080. No añadas `.master("local[*]")`: queremos trabajar con los workers.

In [ ]:
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.appName("Mi-primer-cluster-SBD").getOrCreate()
print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Aplicación:", spark.sparkContext.applicationId)

## 1. Comprueba dónde se ejecuta Python
El nombre jupyter corresponde al driver. Los nombres spark-worker-* proceden de los procesos Python de los executors. El reparto concreto de estas ocho particiones puede variar.

In [ ]:
import socket
print("Driver:", socket.gethostname())
def localizar(particion):
    import socket, time
    time.sleep(0.5)
    yield (socket.gethostname(), sum(particion))
ubicaciones = spark.sparkContext.parallelize(range(80), 8).mapPartitions(localizar).collect()
ubicaciones

## 2. Lee las ventas
La carpeta `/workspace` está montada en todos los contenedores en la misma ruta. Este ejemplo usa un directorio compartido en un único equipo, no almacenamiento distribuido entre portátiles.

In [ ]:
ventas = spark.read.option("header", True).schema("pais STRING, importe INT").csv("/workspace/datos/ventas.csv")
ventas.show()
print("Particiones:", ventas.rdd.getNumPartitions())

## 3. Agrupa y observa la ejecución
El resultado debe ser ES = 30 y FR = 12. Abre http://localhost:4040 mientras la sesión siga activa. Consulta Jobs, Stages y Executors; los recuentos pueden variar por las optimizaciones de Spark.

In [ ]:
totales = ventas.groupBy("pais").agg(F.sum("importe").alias("total"))
totales.orderBy("pais").show()
totales.explain()

## 4. Guarda y recupera el resultado
Solo se sobrescribe la carpeta `salidas/ventas_por_pais` de esta práctica.

In [ ]:
ruta = "/workspace/salidas/ventas_por_pais"
totales.write.mode("overwrite").parquet(ruta)
spark.read.parquet(ruta).show()

## 5. Explica lo que has observado
1. ¿Por qué el master y el driver están en contenedores diferentes?
2. ¿Qué diferencia hay entre un worker y un executor?
3. ¿Por qué este ejemplo puede usar rutas locales compartidas?
4. ¿Dónde consultarías las tareas de la aplicación?
5. ¿Qué cambiaría con el perfil de un solo worker?

In [ ]:
spark.stop()